<a href="https://colab.research.google.com/github/JediMasterKeith/FUNDAI-Laboratories-BASAN/blob/main/Lab3_Game_AI_Basan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: Game AI Using Minimax with Alpha-Beta Pruning

## Fundamentals of Artificial Intelligence

**Name:** Keith Lyndon G. Basan

**Course:** BSCS AI

**Section:** 09282-FUNDAI

**Date:** August 27, 2026

**Selected Game:** Tic-Tac-Toe

**GitHub URL:** https://github.com/JediMasterKeith/FUNDAI-Laboratories-BASAN

## Description
This laboratory implements a Tic-Tac-Toe AI using Minimax with Alpha-Beta
pruning.
The game is playable inside Google Colab using ipywidgets.

In [1]:
import math
import ipywidgets as widgets
from IPython.display import display

In [2]:
class TicTacToeGame:
  X = "X"
  O = "O"
  EMPTY = " "

  def __init__(self): # Re-applying the fix to ensure kernel loads latest definition
    self.board = [self.EMPTY] * 9
    self.current_player = self.X

  def available_moves(self):
    return [i for i, value in enumerate(self.board) if value == self.EMPTY]

  def make_move(self, move):
    self.board[move] = self.current_player
    self.current_player = self.O if self.current_player == self.X else self.X

  def undo_move(self, move):
    self.board[move] = self.EMPTY
    self.current_player = self.O if self.current_player == self.X else self.X

  def get_winner(self):
    winning_lines = [
        (0, 1, 2),
        (3, 4, 5),
        (6, 7, 8),
        (0, 3, 6),
        (1, 4, 7),
        (2, 5, 8),
        (0, 4, 8),
        (2, 4, 6)
    ]

    for a, b, c in winning_lines:
        if self.board[a] != self.EMPTY and self.board[a] == self.board[b] == self.board[c]:
            return self.board[a]
    return None

  def is_draw(self):
    return self.get_winner() is None and len(self.available_moves()) == 0

  def is_terminal(self):
    return self.get_winner() is not None or len(self.available_moves()) == 0

  def utility(self):
    winner = self.get_winner()

    if winner == self.X:
        return 1
    elif winner == self.O:
        return -1
    else:
        return 0

In [3]:
import math

def minimax_alpha_beta(game, alpha=-math.inf, beta=math.inf):
    if game.is_terminal():
        return game.utility(), None

    # MAX player: X
    if game.current_player == TicTacToeGame.X:
        best_value = -math.inf
        best_move = None

        for move in game.available_moves():
            game.make_move(move)
            value, _ = minimax_alpha_beta(game, alpha, beta)
            game.undo_move(move)

            if value > best_value:
                best_value = value
                best_move = move

            alpha = max(alpha, best_value)

            if alpha >= beta:
                break

        return best_value, best_move

    # MIN player: O
    else:
        best_value = math.inf
        best_move = None

        for move in game.available_moves():
            game.make_move(move)
            value, _ = minimax_alpha_beta(game, alpha, beta)
            game.undo_move(move)

            if value < best_value:
                best_value = value
                best_move = move

            beta = min(beta, best_value)

            if alpha >= beta:
                break

        return best_value, best_move

In [4]:
class TicTacToeUI:
    def __init__(self):
        self.game = TicTacToeGame()

        self.buttons = [
            widgets.Button(
                description=" ",
                layout=widgets.Layout(width="60px", height="60px")
            )
            for i in range(9)
        ]

        for i in range(9):
            self.buttons[i].on_click(
                lambda btn, idx=i: self.on_cell_click(idx)
            )

        self.status = widgets.HTML(
            value="<b>Human X moves first.</b>"
        )

        self.reset_button = widgets.Button(
            description="Reset",
            button_style="info"
        )
        self.reset_button.on_click(self.on_reset)

        self.grid = widgets.GridBox(
            children=self.buttons,
            layout=widgets.Layout(
                grid_template_columns="repeat(3, 60px)",
                grid_gap="5px"
            )
        )

        self.widget = widgets.VBox(
            [self.status, self.grid, self.reset_button]
        )

        display(self.widget)

        self.refresh()

    def refresh(self):
        for i, button in enumerate(self.buttons):
            button.description = self.game.board[i]
            button.disabled = (
                self.game.is_terminal()
                or self.game.board[i] != TicTacToeGame.EMPTY
            )

        if self.game.is_terminal():
            winner = self.game.get_winner()

            if winner:
                self.status.value = (
                    f"<b>Player {winner} wins!</b>"
                )
            else:
                self.status.value = "<b>Draw!</b>"
        else:
            self.status.value = (
                f"<b>Current player: {self.game.current_player}</b>"
            )

    def on_cell_click(self, index):
        if self.game.board[index] != TicTacToeGame.EMPTY:
            return

        if self.game.is_terminal():
            return

        if self.game.current_player != TicTacToeGame.X:
            return

        # Human move
        self.game.make_move(index)

        # AI move if game is not finished
        if not self.game.is_terminal():
            ai_move = self.get_ai_move()

            if ai_move is not None:
                self.game.make_move(ai_move)

        self.refresh()

    def get_ai_move(self):
        _, move = minimax_alpha_beta(self.game)
        return move

    def on_reset(self, button):
        self.game = TicTacToeGame()
        self.refresh()


In [5]:
TicTacToeUI()

## Algorithm Explanation

### Minimax
The AI evaluates possible future game states.
Player X maximizes utility, while Player 0 minimizes utility.

### Alpha-Beta Pruning
Alpha-Beta pruning removes branches that cannot change the final decision.
This makes the AI faster while preserving the optimal result.

### Utility
- X wins: +1
- Draw: 0
- 0 wins: -1

## Guide Questions and Answers

### 1. Which player does the AI control?
**Answer:** The AI controls player 'O' (the minimizing player). The human player 'X' moves first, and if the game is not terminal, the AI then makes its move as 'O'.

### 2. What utility values were used?
**Answer:** The utility values used are: +1 for an 'X' win, -1 for an 'O' win, and 0 for a draw.

### 3. How does Alpha-Beta pruning improve Minimax?
**Answer:** Alpha-Beta pruning significantly improves Minimax by eliminating branches from the search tree that cannot possibly influence the final decision. This dramatically reduces the number of game states that need to be evaluated, making the AI faster while still guaranteeing that the optimal move is found.

### 4. What happens when the human chooses a move that leads to a draw?
**Answer:** When the human (Player X) chooses a move that leads to a draw, the AI (Player O), using Minimax with Alpha-Beta pruning, will aim for the draw. A draw has a utility of 0 for both players, which is the best outcome for Player O if it cannot win (utility -1 if O loses), and better than losing for Player X. Therefore, the AI will play optimally to secure the draw.

### 5. Why is Tic-Tac-Toe suitable for full Minimax search?
**Answer:** Tic-Tac-Toe is suitable for a full Minimax search because of its small game tree and limited number of possible states. The game is simple enough that even with a full search (exploring all possible moves to the game's conclusion), the computation remains feasible and fast, allowing the AI to always choose the optimal move.

## Reflection

### Challenges Encountered
Implementing this Tic-Tac-Toe AI presented several challenges. Primarily, I focused on understanding and correctly implementing the recursive nature of the Minimax algorithm. A significant part of this involved integrating the Alpha-Beta pruning logic to efficiently cut off unproductive branches in the search tree, which required careful thought to ensure the pruning rules were applied correctly. Furthermore, ensuring the `make_move` and `undo_move` operations correctly managed the game state, especially during the recursive calls, was crucial for the algorithm's correctness. Finally, debugging the overall game flow and the interactions of buttons within the `ipywidgets` framework required attention to detail to achieve a smooth user experience.

### What I Learned
Through this laboratory, I gained a deep understanding of the Minimax algorithm and its practical application in game AI. I learned how to effectively optimize search algorithms using Alpha-Beta pruning, which significantly improved the computational efficiency without sacrificing optimality. This project also provided valuable experience in designing a robust game state representation and implementing core game logic. Lastly, it offered hands-on exposure to `ipywidgets` for creating interactive user interfaces directly within Google Colab, enhancing my skills in developing interactive analytical tools.